In [ ]:
import os 
from pathlib import Path

from transformers import pipelines

from trouver.helper.html import (
    add_HTML_tag_data_to_raw_text, add_space_to_lt_symbols_without_space, remove_html_tags_in_text)

from trouver.obsidian.file import MarkdownFile, MarkdownLineEnum
from trouver.obsidian.vault import VaultNote

from trouver.machine_learning.tokenize.def_and_notat_token_classification import def_and_notat_preds_by_model, convert_double_asterisks_to_html_tags, _add_nice_boxing_attrs_to_def_and_notat_tags, get_def_and_notat_predictions, _collate_html_tags, _get_main_text_lines




In [ ]:
from unittest import mock
import shutil
import tempfile

from fastcore.test import *

from trouver.helper.tests import _test_directory

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
from trouver.helper.latex.core import get_dollar_sign_syntax_errors

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _format_main_text_and_add_html_tag_data(
        note: VaultNote,
        pipeline: pipelines.token_classification.TokenClassificationPipeline, # The token classification pipeline that is used to predict whether tokens are part of definitions or notations introduced in the text.
        add_boxing_attr_to_existing_def_and_notat_markings: bool,
        excessive_space_threshold: int,
        main_text: str,  # The main text to format and to add HTML tag data to
        ) -> str:
    """Helper function to `auto_mark_def_and_notats`"""
    # 1. Pre-processing
    main_text = add_space_to_lt_symbols_without_space(main_text)
    main_text = convert_double_asterisks_to_html_tags(main_text)
    main_text, existing_html_tag_data = remove_html_tags_in_text(main_text)

    # 2. Handle existing tag styling
    if add_boxing_attr_to_existing_def_and_notat_markings:
        existing_html_tag_data = _add_nice_boxing_attrs_to_def_and_notat_tags(
            existing_html_tag_data)

    # 3. Get New Predictions
    new_tag_data = get_def_and_notat_predictions(
        main_text, pipeline, excessive_space_threshold, note)

    # 4. Merge old and new tags
    all_tags_to_add = _collate_html_tags(
        existing_html_tag_data, new_tag_data)

    # 5. Apply to text
    return add_HTML_tag_data_to_raw_text(main_text, all_tags_to_add)
    # html_tags_to_add = _get_token_preds_by_dividing_main_text(
    #     main_text, pipeline, note, excessive_space_threshold)

    # html_tags_to_add_back = _collate_html_tags(
    #     existing_html_tag_data, html_tags_to_add)
    # return add_HTML_tag_data_to_raw_text(main_text, html_tags_to_add_back)


In [ ]:
from unittest.mock import MagicMock

class MockTokenizer:
    """Mocks a HuggingFace tokenizer."""
    model_max_length = 512
    
    def __call__(self, text):
        # CORRECTED: Return a dict, not SimpleNamespace
        length = len(text) // 5 + 1
        return {'input_ids': [0] * length}

# Update the pipeline to use the corrected tokenizer
class MockPipeline:
    """Mocks the TokenClassificationPipeline."""
    def __init__(self, preds):
        self.tokenizer = MockTokenizer()
        self._preds = preds

    def __call__(self, text):
        return self._preds

In [ ]:

#| hide
#| notest
from types import SimpleNamespace
from unittest.mock import MagicMock

# --- Test Setup ---

# 1. Setup a Mock Note
# Use MagicMock so .path() is callable if needed, or .path works as a property
mock_note = MagicMock()
mock_note.name = "Test Note"
mock_note.path.return_value = "test/path/Test Note.md" 

# 2. Setup Input Text
# "The Galois group $\operatorname{Gal}(L/K)$ of the extension L/K is..."
# Indices:
# T=0, h=1, e=2,  =3
# G=4, a=5, l=6, o=7, i=8, s=9,  =10 (Galois)
# g=11, r=12, o=13, u=14, p=15 (group)
#  =16
# $=17, \=18 ... $\operatorname{Gal}(L/K)$ ends at 42
text_input = r"The Galois group $\operatorname{Gal}(L/K)$ of the extension L/K is..."

# 3. Setup Mock Predictions
# We simulate the model finding:
# A. 'Galois group' as a definition.
# B. 'Gal' (inside the latex) as a notation.
mock_preds = [
    # Definition: "Galois group" (indices 4-16)
    {
        'entity': 'B-definition', 'score': 0.99, 'index': 1, 
        'word': 'Galois', 'start': 4, 'end': 10
    },
    {
        'entity': 'I-definition', 'score': 0.99, 'index': 2, 
        'word': 'group', 'start': 11, 'end': 16
    },
    # Notation: Model finds 'Gal' inside '$\operatorname{Gal}(L/K)$'
    # We simulate it finding the substring at index 31-34 (inside the operatorname)
    {
        'entity': 'B-notation', 'score': 0.98, 'index': 3, 
        'word': 'Gal', 'start': 31, 'end': 34
    }
]

mock_pipeline = MockPipeline(mock_preds)

# --- Run Function ---
formatted_text = _format_main_text_and_add_html_tag_data(
    note=mock_note,
    pipeline=mock_pipeline,
    add_boxing_attr_to_existing_def_and_notat_markings=True,
    excessive_space_threshold=2,
    main_text=text_input
)

# --- Assertions ---

# 1. Check if "Galois group" is wrapped in a definition tag
# It should be boxed because we set add_boxing...=True
# Note: exact tag name (b vs span) depends on your implementation.
# We check for the key components.
assert 'Galois group' in formatted_text
assert 'definition=""' in formatted_text
assert 'style="border-width:1px;border-style:solid;padding:3px"' in formatted_text

# 2. Check if $\operatorname{Gal}(L/K)$ is wrapped in a notation tag
# The logic should have expanded 'Gal' to the full math string.
expected_notation_str = r'$\operatorname{Gal}(L/K)$'
expected_notation_html = f'<span notation="" style="border-width:1px;border-style:solid;padding:3px">{expected_notation_str}</span>'

assert expected_notation_html in formatted_text, \
    f"Notation tag missing or incorrect.\nExpected: {expected_notation_html}\nGot: {formatted_text}"

# --- Test Case 2: Handling Existing Tags ---
# Text with existing "Extension field L" definition
text_with_existing = r'The <b definition="">extension field $L$</b> is...'
mock_pipeline_empty = MockPipeline([]) 

formatted_text_2 = _format_main_text_and_add_html_tag_data(
    note=mock_note,
    pipeline=mock_pipeline_empty,
    add_boxing_attr_to_existing_def_and_notat_markings=True, 
    excessive_space_threshold=2,
    main_text=text_with_existing
)

# Check if the 'style' attribute was added (boxing) to the existing bold tag
assert 'style="border-width:1px;border-style:solid;padding:3px"' in formatted_text_2, \
    "Boxing attributes were not added to existing tag."
assert '<b definition="" style=' in formatted_text_2

# --- Test Case 3: Double Asterisks Conversion ---
# The function calls `convert_double_asterisks_to_html_tags` internally.
# Example: **$K$**
text_asterisks = r"Let **$K$** be a field."
formatted_text_3 = _format_main_text_and_add_html_tag_data(
    note=mock_note,
    pipeline=mock_pipeline_empty,
    add_boxing_attr_to_existing_def_and_notat_markings=False,
    excessive_space_threshold=2,
    main_text=text_asterisks
)

# Should convert **...** to a tag (definition or notation depending on content/heuristics)
# Usually math mode inside ** -> notation.
assert '<span notation="">$K$</span>' in formatted_text_3 or '<b definition="">$K$</b>' in formatted_text_3

C:\Users\hyunj\Documents\Development\Python\trouver\trouver\helper\html.py:99: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  parsed_soup = BeautifulSoup(text, 'html.parser')


In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def _write_text_with_html_tag_preds_to_note(
        note: VaultNote,
        mf: MarkdownFile,
        main_text: str,
        first_non_metadata_line: int,
        see_also_line: int
        ) -> None:
    """
    Final step of the marking process; the new contents of the note are written.

    Helper function to `auto_mark_def_and_notats`
    """
    mf.remove_lines(first_non_metadata_line, see_also_line)
    mf.insert_line(first_non_metadata_line,
                   {'type': MarkdownLineEnum.DEFAULT, 'line': main_text})
    mf.add_tags('_auto/def_and_notat_identified')
    mf.write(note)

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification
def auto_mark_def_and_notats(
        note: VaultNote,  # The standard information note in which to find the definitions and notations.
        pipeline: pipelines.token_classification.TokenClassificationPipeline, # The token classification pipeline that is used to predict whether tokens are part of definitions or notations introduced in the text.
        # remove_existing_def_and_notat_markings: bool = False,  # If `True`, remove definition and notation markings (both via surrounding by double asterisks `**` as per the legacy method and via HTML tags)
        excessive_space_threshold: int = 2,
        add_boxing_attr_to_existing_def_and_notat_markings: bool = True, # If `True`, then nice attributes are added to the existing notation HTML tags, if not already present.
        check_for_dollar_sign_syntax_errors_first: bool = False,
    ) -> None:
    """
    Predict and mark where definitions and notation occur in a note using
    a token classification ML model.

    Assumes that the note is a standard information note that does not
    have a lot of "user modifications", such as footnotes, links,
    and HTML tags. If
    there are many modifications, then these might be deleted.

    Assumes that the paragraphs in the text of the note are "not too long".
    Currently, this means that the paragraphs in the number of tokens
    in the text of the note should (roughly) not exceed 
    `pipeline.tokenizer.model_max_length`.

    Existing markings for definition and notation data (i.e. by
    surrounding with double asterisks or by HTML tags) are preserved
    (and turned into HTML tags), unless the markings overlap with 
    predictions, in which case the original is preserved (and still
    turned into an HTML tag if possible)

    Since the model can make "invalid" predictions (mostly those which
    start or end within a LaTeX math mode str), the actual markings
    are not necessarily direct translates from the model's predictions.
    See the helper function `_consolidate_token_preds` for more details
    on how this is implemented.
    
    **Raises**
    Warning messages (`UserWarning`) are printed in the following situations:

    - There are two consecutive tokens within the `pipeline`'s predictions
      of different entity types (e.g. one is predicted to belong within a
      definition and the other within a notation), but the latter token's
      predicted `'entity'` more specifically begins with `'I-'` (i.e. is
      `'I-definition'` or `'I-notation'`) as opposed to `'B-'`.
        - `note`'s name, and path are included in the warning message in
          this case.
    - There are two consecutive tokens within the `pipeline`'s predictions
      which the pipeline predicts to belong to the same entity, and yet
      there is excessive space (specified by `excessive_space_threshold`)
      between the end of the first token and the start of the second.

    """
    mf = MarkdownFile.from_vault_note(note)
    mf.cleanup_formatting()
    # _process_mf(mf)
    first_non_metadata_line, see_also_line = _get_main_text_lines(mf)
     
    main_text = mf.text_of_lines(first_non_metadata_line, see_also_line)
    # --- Dollar Sign Syntax Safety Gate ---
    if check_for_dollar_sign_syntax_errors_first:
        # We don't check spacing by default here as it's often too strict for Obsidian
        dollar_sign_syntax_errors = get_dollar_sign_syntax_errors(main_text)
        if dollar_sign_syntax_errors:
            error_details = "\n- ".join(dollar_sign_syntax_errors)
            warnings.warn(
                f"Skipping auto-marking for '{note.name}' due to LaTeX syntax errors:\n- {error_details}"
            )
            return # Exit early to prevent malformed HTML tag indices

    main_text = _format_main_text_and_add_html_tag_data(
        note, pipeline, add_boxing_attr_to_existing_def_and_notat_markings,
        excessive_space_threshold, main_text)
    _write_text_with_html_tag_preds_to_note(
        note, mf, main_text, first_non_metadata_line, see_also_line)


In [ ]:
#| hide
from transformers import BatchEncoding, pipelines, PreTrainedTokenizer, PreTrainedTokenizerFast
from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer

In [ ]:
#| notest
# 1. Load the Model
# Replace 'hyunjongkim/math-def-and-notat-token-classification' with your actual model path/ID
# We use 'token-classification' task. 'aggregation_strategy="simple"' helps merge B- and I- tags automatically.
model = AutoModelForTokenClassification.from_pretrained('hyunjongkimmath/def_and_notat_token_classification_model_modernbert_base')
tokenizer = AutoTokenizer.from_pretrained('hyunjongkimmath/def_and_notat_token_classification_model_modernbert_base')
def_notat_classifier = pipeline('ner', model=model, tokenizer=tokenizer)

from pathlib import Path
note = VaultNote(vault=Path(r'C:/Users/hyunj/Documents/Obsidian/Math/algebraic_geometry/abelian_varieties/bosch_lutkebohmert_raynaud_nm'), rel_path=r'chapter_3_the_smoothening_process/36_algebraic_approximation_of_formal_points/bosch_lutkebohmert_raynaud_nm_Lemma 13.md')
# note = VaultNote(vault=Path(r'C:/Users/hyunj/Documents/Obsidian/Math/algebraic_geometry/abelian_varieties/bosch_lutkebohmert_raynaud_nm'), rel_path=r'chapter_3_the_smoothening_process/31_statement_of_the_theorem/bosch_lutkebohmert_raynaud_nm_Definition 1_page_60.md')

# A dummy note object (required by the function for logging)
# note = SimpleNamespace(name="Example Note", path="Example.md")

# 3. Run Prediction and Formatting
auto_mark_def_and_notats(note, def_notat_classifier, check_for_dollar_sign_syntax_errors_first=True)
# formatted_text = predict_and_mark_def_and_notats(
#     main_text=text,
#     pipeline=def_notat_classifier,
#     # note=note,
#     formatter=latex_highlight_formatter,
#     excessive_space_threshold=2
# )

# 4. View Result
print(note.text())


Device set to use cpu
Compiling the model with `torch.compile` and using a `torch.cpu` device is not supported. Falling back to non-compiled mode.


---
cssclass: clean-embeds
aliases: []
tags: [_auto/_meta/concept, _auto/_meta/proof, _auto/_meta/TODO/split, _meta/literature_note, _auto/_meta/notation, _auto/def_and_notat_identified, _reference/bosch_lutkebohmert_raynaud_nm]
---
# Topic[^1]
Lemma 13. Let $f_0$ be a global section of $\mathcal{O}_{\text {Ay }}$ such that $\sigma^* f_0$ does not vanish at $\hat{\eta}$. Then there exists a commutative diagram of $S$-morphisms

84

3. The Smoothening Process

such that $V^{\prime}$ is smooth over $S$ and such that $\tau^* f_0$ divides each $\tau^* f_i, i=1, \ldots, r$, in $\Gamma\left(V^{\prime}, \mathcal{O}_{V^{\prime}}\right)$

In the proof of the lemma, we will use WeierstraB division for the formal power series ring $\hat{R}\left[\left[T_1, \ldots, T_n\right]\right]$; cf. Bourbaki $[2]$, Chap. VII, $\S 3, n^{\circ} 8$. Let us first recall divisor in $T_n$ of degree $d \geqq 0$ if the coefficients $a_v \in \hat{R}\left[\left[T_1, \ldots, T_{n-1}\right]\right]$ of the power series ex

In the following examples, we mock pipeline objects instead of using actual ones.

In the below example, we run the `auto_mark_def_and_notats` function on a note that has double asterisks `**` surrounding parts of the text that introduced definitions or notations. In these cases, appropriate HTML tags replace the double asterisks instead.

In [ ]:
with (tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir,
      mock.patch('__main__.pipelines.token_classification.TokenClassificationPipeline') as mock_pipeline):
    temp_vault = Path(temp_dir) / 'test_vault_6'
    shutil.copytree(_test_directory() / 'test_vault_6', temp_vault)

    mock_pipeline.tokenizer.model_max_length = 512

    vn = VaultNote(temp_vault, name='reference_with_tag_labels_Definition 2')
    print("Text before:\n\n")
    print(vn.text())
    print("\n\n\nText after:\n")
    auto_mark_def_and_notats(vn, mock_pipeline)
    print(vn.text())
    mf = MarkdownFile.from_vault_note(vn)
    assert mf.has_tag('_auto/def_and_notat_identified')



Text before:


---
cssclass: clean-embeds
aliases: []
tags: [_meta/literature_note, _meta/definition, _meta/notation]
---
# Ring of integers modulo $n$[^1]

Let $n \geq 1$ be an integer. The **ring of integers modulo $n$**, denoted by **$\mathbb{Z}/n\mathbb{Z}$**, is, informally, the ring whose elements are represented by the integers with the understanding that $0$ and $n$ are equal.

More precisely, $\mathbb{Z}/n\mathbb{Z}$ has the elements $0,1,\ldots,n-1$.

...


# See Also
- [[reference_with_tag_labels_Exercise 1|reference_with_tag_labels_Z_nZ_is_a_ring]]
# Meta
## References

## Citations and Footnotes
[^1]: Kim, Definition 2



Text after:

---
cssclass: clean-embeds
aliases: []
tags: [_meta/definition, _auto/def_and_notat_identified, _meta/literature_note, _meta/notation]
---
# Ring of integers modulo $n$[^1]

Let $n \geq 1$ be an integer. The <b definition="" style="border-width:1px;border-style:solid;padding:3px">ring of integers modulo $n$</b>, denoted by <span notation="" s

In [ ]:
# TODO: more examples with pipeline mocking actual outputs